# Reconstructing Unique Runner IDs from Parkrun Data

Can we reconstruct a reliable unique runner identifier from ~9M scraped parkrun results that have no athlete ID? See [runner_id_report.md](runner_id_report.md) for the full write-up.

In [69]:
from parkrun.data import load_data

df = load_data()
print(f"Rows: {len(df):,}")
print(f"Unique names: {df['Name'].nunique():,}")
print(f"Most common names:")
print(df["Name"].value_counts().head(10).to_string())

Rows: 9,040,767
Unique names: 902,003
Most common names:
Name
Paul SMITH        1620
David SMITH       1559
Mark SMITH        1445
Andrew SMITH      1376
Chris JONES       1267
Richard SMITH     1203
Mark JONES        1186
David JONES       1166
David WILLIAMS    1163
David BROWN       1120


## Key discovery: `Total parkruns` is a static fingerprint

`Total parkruns` is **not** an incrementing counter — it's a snapshot captured at scrape time. Every row for the same runner shows the same value, making it a per-runner constant.

In [70]:
# Demonstrate: a runner with a unique name has constant Total parkruns across all dates
runner = df[df["Name"] == "Aaron BENTLEY"].sort_values("Event Date")
print(
    f"Aaron BENTLEY: {len(runner)} rows, Total parkruns always = {runner['Total parkruns'].unique()}"
)
print(
    runner[["Event Date", "Total parkruns", "Event Name", "Age Group", "Time"]]
    .head(8)
    .to_string()
)

# 14 different "Paul SMITHs" ran on 1 Jan 2025, each with a different Total parkruns
ps = df.query("Name == 'Paul SMITH'")
ps_jan1 = ps[ps["Event Date"] == ps["Event Date"].min()]
print(f"\nPaul SMITH on {ps_jan1['Event Date'].iloc[0]}: {len(ps_jan1)} runners")
print(
    ps_jan1[["Total parkruns", "Event Name", "Age Group", "Time"]]
    .sort_values("Total parkruns")
    .to_string(index=False)
)

Aaron BENTLEY: 30 rows, Total parkruns always = <IntegerArray>
[51]
Length: 1, dtype: Int64
         Event Date  Total parkruns                    Event Name Age Group   Time
2402690  2025-01-11              51             Mote Park parkrun   VM35-39  25:09
2551871  2025-01-25              51  Maidstone River Park parkrun   VM35-39  22:48
4648187  2025-02-08              51  Maidstone River Park parkrun   VM35-39  23:02
7523236  2025-03-01              51  Maidstone River Park parkrun   VM35-39  22:57
3151058  2025-03-08              51  Maidstone River Park parkrun   VM35-39  23:45
2476268  2025-03-22              51          Clumber Park parkrun   VM35-39  23:21
8468342  2025-03-29              51  Maidstone River Park parkrun   VM35-39  23:03
9056550  2025-04-05              51  Maidstone River Park parkrun   VM35-39  21:22

Paul SMITH on 2025-01-01: 14 runners
 Total parkruns                Event Name Age Group  Time
             10 Polkemmet Country parkrun   VM50-54 34:03
       

## Collision analysis

**(Name, Gender, Total parkruns)** is the base fingerprint. How often does it collide? A runner can only attend one parkrun per Saturday, so same-day appearances at different events reveal collisions.

In [71]:
key_cols = ["Name", "Gender", "Total parkruns"]
n_groups = df.groupby(key_cols).ngroups
print(f"Unique (Name, Gender, Total parkruns) groups: {n_groups:,}")

# Same-day conflicts: rows sharing the key that appear on the same date
group_dates = df.groupby(key_cols + ["Event Date"]).size().reset_index(name="count")
conflicts = group_dates[group_dates["count"] > 1]
conflict_keys = conflicts[key_cols].drop_duplicates()
print(f"Groups with same-day conflicts: {len(conflict_keys):,}")

# Classify: duplicate rows vs genuine collisions (different events)
dupes = df.merge(conflicts[key_cols + ["Event Date"]], on=key_cols + ["Event Date"])


def classify(g):
    if g["Event Name"].nunique() == 1:
        return "duplicate_row" if g["Time"].nunique() == 1 else "same_event_diff_time"
    return "different_events"


print("\nConflict breakdown:")
print(
    dupes.groupby(key_cols + ["Event Date"]).apply(classify).value_counts().to_string()
)

# First-timers (Total parkruns=1) each get their own ID
ft = df[df["Total parkruns"] == 1]
print(f"\nFirst-timer rows (each = distinct runner): {len(ft):,}")

Unique (Name, Gender, Total parkruns) groups: 1,262,911
Groups with same-day conflicts: 11,547

Conflict breakdown:
different_events        11820
duplicate_row            5325
same_event_diff_time      506

First-timer rows (each = distinct runner): 182,845


## Assign runner IDs

The algorithm in `parkrun/runner_id.py` works in 3 stages:
1. **Base grouping** by (Name, Gender, Total parkruns) → ~1.26M groups
2. **First-timer expansion** — each `Total parkruns = 1` row is a distinct runner
3. **Conflict splitting** — greedy partitioning using same-day + age-group incompatibility signals

In [72]:
# ── Assign Runner IDs ──────────────────────────────────────────────
from parkrun.runner_id import assign_runner_ids, validate_runner_ids, print_validation
import time

t0 = time.time()
runner_id = assign_runner_ids(df)
elapsed = time.time() - t0

print(f"Assigned runner IDs in {elapsed:.1f}s")
print(f"Total rows:     {len(runner_id):>12,}")
print(f"Unique runners: {runner_id.nunique():>12,}")

df["Runner_ID"] = runner_id

Assigned runner IDs in 19.8s
Total rows:        9,040,767
Unique runners:    1,296,087


In [64]:
# ── Validation ─────────────────────────────────────────────────────
results = validate_runner_ids(df, runner_id)
print_validation(results)

Runner ID Validation Results
  [PASS] No runner ID assigned to different events on the same date
  [PASS] Each Runner_ID maps to exactly one Name
  [PASS] Each Runner_ID maps to exactly one Gender
  [PASS] Each Runner_ID maps to exactly one Total parkruns value
  [FAIL (878 violations)] Each Runner_ID has at most 2 age groups (birthday transition)
  [PASS] Runners with Total parkruns=1 have exactly 1 row

  Total rows:         9,040,767
  Unique runners:     1,296,091
  Avg rows/runner:         6.98


In [ ]:
# ── Spot-check: trace a common name through the IDs ───────────────
paul_smiths = df[df["Name"] == "Paul SMITH"].sort_values(["Runner_ID", "Event Date"])
n_ids = paul_smiths["Runner_ID"].nunique()
print(f"'Paul SMITH' → {n_ids} distinct Runner_IDs across {len(paul_smiths)} rows\n")

# Show a few runner slices
for rid, group in paul_smiths.groupby("Runner_ID"):
    if len(group) < 3:
        continue
    g = group[["Event Date", "Event Name", "Age Group", "Time"]].head(3)
    print(
        f"Runner_ID {rid}  ({len(group)} rows, Age Group: {group['Age Group'].iloc[0]})"
    )
    print(g.to_string(index=False))
    print()
    if rid > paul_smiths["Runner_ID"].unique()[4]:
        break

'Paul SMITH' → 137 distinct Runner_IDs across 1620 rows

Runner_ID 1254638  (13 rows, Age Group: SM25-29)
Event Date         Event Name Age Group  Time
2025-01-01 Birkenhead parkrun   SM25-29 26:28
2025-01-04 Birkenhead parkrun   SM25-29 24:07
2025-01-18 Birkenhead parkrun   SM25-29 23:12

Runner_ID 1254639  (4 rows, Age Group: VM65-69)
Event Date             Event Name Age Group  Time
2025-10-11 Wanstead Flats parkrun   VM65-69 31:39
2025-10-18 Wanstead Flats parkrun   VM65-69 30:54
2025-10-25 Wanstead Flats parkrun   VM65-69 34:49

Runner_ID 1254640  (11 rows, Age Group: SM30-34)
Event Date            Event Name Age Group  Time
2025-02-01       Penrith parkrun   SM30-34 18:47
2025-02-08          York parkrun   SM30-34 18:02
2025-02-15 South Shields parkrun   SM30-34 19:01

Runner_ID 1254644  (4 rows, Age Group: SM30-34)
Event Date                 Event Name Age Group  Time
2025-02-15 Leicester Victoria parkrun   SM30-34 41:16
2025-03-15     Lloyd parkrun, Croydon   SM30-34 44:27
2025